# 25 · App configuration, compaction and caching

App groups a root agent, plugins and runtime policies. Compaction summarizes old conversation events; context caching can reuse large stable prefixes but also has storage costs.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** 1 Lite call by default; cache construction only. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.apps import App, ResumabilityConfig
from google.adk.apps.app import EventsCompactionConfig, ContextCacheConfig
agent = llm("app_shop", "Answer ecommerce questions briefly.")
app = App(name="course_app", root_agent=agent,
          resumability_config=ResumabilityConfig(is_resumable=True),
          events_compaction_config=EventsCompactionConfig(compaction_interval=3, overlap_size=1))
lab = await make_lab(app=app)
await ask(lab, "My order is O1001. Acknowledge it.")
print("App:", app.name, "resumable:", app.resumability_config.is_resumable)


In [ ]:
# Construction-only: attaching this to App can create billable server caches.
cache_config = ContextCacheConfig(min_tokens=32768, cache_intervals=5, ttl_seconds=300)
print(cache_config.model_dump())


Resumability needs persisted session events and an invocation ID when continuing an interrupted run. It does not make external side effects exactly-once; use idempotency keys. This notebook does not reach the compaction threshold or enable a billable cache by default.


## Try it

Use SQLite sessions from notebook 05 and send four short messages to observe compaction. Account for the extra summarization calls.


## Reference

[Official ADK documentation](https://adk.dev/apps/). Shared configuration: `config.json`. No environment activation or login cells are needed.
